# CrewAI Samples

Willkommen! In diesem Notebook kannst du alle CrewAI-Beispiele direkt ausführen.

**So benutzt du dieses Notebook:**
- Klicke auf eine Zelle und drücke **Shift+Enter** um sie auszuführen
- Führe die Zellen immer **von oben nach unten** aus
- Das ▶ Symbol links neben einer Zelle zeigt ob sie gerade läuft

---

## Setup

Diese Zelle muss **einmalig am Anfang** ausgeführt werden.
Sie lädt den API-Key und prüft ob alles funktioniert.

In [ ]:
import os
from dotenv import load_dotenv

# Lädt OPENAI_API_KEY aus der .env Datei
load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    print("✓ API-Key gefunden - bereit!")
else:
    print("✗ Kein API-Key gefunden! Bitte .env Datei prüfen.")

---
## Sample 1 - Ein einzelner Agent (+ Bildgenerierung)

Das einfachste CrewAI-Programm, jetzt mit zwei Agenten:

Ein **Agent** ist wie eine KI-Person mit:
- einer **Rolle** (z.B. "Tier-Experte")
- einem **Ziel** (was will er erreichen?)
- einer **Hintergrundgeschichte** (das macht die KI besser in ihrer Rolle)

Agenten können auch **Tools** benutzen – zum Beispiel DALL-E zum Bilder erzeugen! 🎨

### Ausprobieren:
Ändere `TIER` auf ein beliebiges Tier - auch Fantasietiere funktionieren! 🐉

In [ ]:
import json
from crewai import Agent, Task, Crew
from crewai_tools import DallETool
from IPython.display import display, Markdown

# ── Hier kannst du das Tier eingeben ─────────────────────
TIER = input("Welches Tier soll beschrieben werden? (z.B. Axolotl, Drache, Krake): ").strip() or "Axolotl"
# ─────────────────────────────────────────────────────────

# ── Schritt 1: Agenten erstellen ─────────────────────────

tier_experte = Agent(
    role="Tier-Experte und Naturforscher",
    goal="Erstelle faszinierende und lehrreiche Steckbriefe ueber Tiere.",
    backstory="""        Du bist ein begeisterter Naturforscher und Biologe mit 20 Jahren Erfahrung.
        Du liebst es, Kindern und Jugendlichen die Tierwelt naeher zu bringen.
        Deine Steckbriefe sind immer spannend, voller ueberraschender Fakten
        und auch fuer Fantasietiere voller Kreativitaet.
    """,
    verbose=False,
)

# Der Bild-Agent hat das DallE-Tool – damit kann er Bilder erzeugen!
bild_agent = Agent(
    role="Illustrator und Bildkuenstler",
    goal="Erzeuge ein farbenfrohes, ansprechendes Bild das perfekt zum Tier passt.",
    backstory="""        Du bist ein kreativer Illustrator mit einem Fable fuer Tiere und Natur.
        Du verwandelst Beschreibungen in lebendige, detailreiche Bilder.
        Dein Stil ist bunt, freundlich und fuer Kinder und Jugendliche geeignet.
    """,
    tools=[DallETool(model="dall-e-3", size="1024x1024", quality="standard")],
    verbose=False,
)

# ── Schritt 2: Aufgaben erstellen ────────────────────────

steckbrief_aufgabe = Task(
    description=f"""    
        Erstelle einen kreativen Steckbrief fuer: {TIER}

        Der Steckbrief soll enthalten:
        - Name und Herkunft
        - Aussehen (wie sieht es aus?)
        - Besondere Faehigkeiten oder Superkraefte
        - Ernaehrung (was frisst es?)
        - Ein ueberraschendes Fakt das kaum jemand kennt

        Schreibe spannend und fuer 12-14 jaehrige Schueler geeignet.
        Falls es ein Fantasietier ist, erfinde kreative aber glaubwuerdige Details.
    """,
    expected_output="""        Ein vollstaendiger, spannend geschriebener Tier-Steckbrief
        mit allen genannten Punkten, formatiert mit Ueberschriften.
    """,
    agent=tier_experte,
)

bild_aufgabe = Task(
    description=f"""    
        Erzeuge ein farbenfrohes, ansprechendes Bild von: {TIER}

        Nutze den Steckbrief des vorherigen Agenten als Inspiration fuer den Bildstil.
        Das Bild soll das Tier in seiner natuerlichen Umgebung zeigen,
        freundlich und fuer Kinder geeignet, im Stil einer Naturillustration.

        Gib die URL des generierten Bildes zurueck.
    """,
    expected_output="Eine URL die direkt auf das generierte Bild zeigt.",
    agent=bild_agent,
    context=[steckbrief_aufgabe],
)

# ── Schritt 3: Crew starten ──────────────────────────────

crew = Crew(
    agents=[tier_experte, bild_agent],
    tasks=[steckbrief_aufgabe, bild_aufgabe],
    verbose=False,
)

crew.kickoff()

# Steckbrief anzeigen
display(Markdown(f"""
---
## Steckbrief: {TIER}

{steckbrief_aufgabe.output.raw}
"""))

# Bild-URL aus dem JSON-Ergebnis des DallE-Tools holen und als Link anzeigen
try:
    bild_daten = json.loads(bild_aufgabe.output.raw)
    bild_url = bild_daten.get("image_url", bild_aufgabe.output.raw)
except (json.JSONDecodeError, AttributeError):
    bild_url = bild_aufgabe.output.raw

display(Markdown(f"""
---
## Bild: {TIER}

[Bild öffnen]({bild_url})
"""))

### 💡 Experimentiere!

Verändere den Code und führe die Zelle erneut aus:

1. **Anderes Tier:** Ändere `TIER = "Axolotl"` zu einem anderen Tier
2. **Andere Rolle:** Ändere `role="Tier-Experte..."` - was passiert wenn der Agent ein "Komiker" ist?
3. **Andere Sprache:** Füge zur `description` hinzu: `"Schreibe auf Englisch."`
4. **Fantasietier erfinden:** Probiere `TIER = "Feuerdrache"` oder `TIER = "Unsichtbare Katze"`

---
## Sample 2 – Das Problem mit veralteten Daten

KI-Modelle werden zu einem bestimmten Zeitpunkt **trainiert** und lernen danach nichts Neues dazu.
Alles was nach dem Trainings-Datum passiert ist, weiss die KI **nicht**.

Das nennt man den **Knowledge Cutoff** (Wissens-Grenze).

### Der Vergleich:
Wir fragen einen Agenten nach **aktuellen Wahlumfragen** für Deutschland –
einmal **ohne** Internetzugang, einmal **mit**.

---
### Teil A – Agent ohne Web-Tool ❌
Nur Trainingswissen. Beobachte: Der Agent nennt veraltete Zahlen oder gibt zu, dass er keine aktuellen Daten hat.

In [ ]:
from crewai import Agent, Task, Crew
from IPython.display import display, Markdown

# Agent OHNE Tool - nur Trainingswissen
agent_ohne_tool = Agent(
    role="Politischer Analyst",
    goal="Recherchiere aktuelle Wahlumfragen und politische Stimmungsbilder in Deutschland.",
    backstory="""\
        Du bist ein erfahrener politischer Analyst mit Fokus auf deutsche Politik.
        Du wertest Wahlumfragen aus und erklaerst politische Trends verstaendlich.
        Du bist ehrlich wenn deine Informationen moeglicherweise nicht aktuell sind.
    """,
    verbose=False,
    # Kein tools=[] --> kein Internetzugang!
)

aufgabe_ohne_tool = Task(
    description="""\
        Erstelle eine Uebersicht der aktuellen Wahlumfragen fuer Deutschland.

        Beantworte:
        1. Welche Umfragewerte haben CDU/CSU, SPD, Gruene, Linke, FDP, AfD, BSW aktuell?
        2. Welche Partei liegt vorne?
        3. Von wann sind deine Informationen?

        Nenne konkrete Prozentzahlen und das Datum der Umfragen.
    """,
    expected_output="Uebersicht der Wahlumfragen mit Prozentzahlen, Datum und Quellenangabe.",
    agent=agent_ohne_tool,
)

ergebnis_ohne = Crew(agents=[agent_ohne_tool], tasks=[aufgabe_ohne_tool]).kickoff()

display(Markdown(f"---\n## Ergebnis: Ohne Web-Tool\n\n{ergebnis_ohne.raw}"))

---
### Teil B – Agent mit Web-Tool ✅

Jetzt bekommt der **gleiche Agent** ein Google-Such-Tool.
Das ist der **einzige Unterschied** zum Code oben: `tools=[such_tool]`

**Voraussetzung:** `SERPER_API_KEY` in der `.env` Datei.  
Kostenlos registrieren (2500 Suchen gratis): [serper.dev](https://serper.dev)

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, WebsiteSearchTool
from IPython.display import display, Markdown

# Das Web-Such-Tool - liest SERPER_API_KEY automatisch aus .env
such_tool = SerperDevTool()
#such_tool = WebsiteSearchTool(website='https://dawum.de/')

# Identischer Agent - nur tools=[such_tool] ist neu!
agent_mit_tool = Agent(
    role="Politischer Analyst",
    goal="Recherchiere aktuelle Wahlumfragen und politische Stimmungsbilder in Deutschland.",
    backstory="""\
        Du bist ein erfahrener politischer Analyst mit Fokus auf deutsche Politik.
        Du wertest Wahlumfragen aus und erklaerst politische Trends verstaendlich.
        Du nutzt immer aktuelle Quellen und gibst diese an.
    """,
    verbose=True,
    tools=[such_tool],   # <-- das ist der einzige Unterschied!
)

aufgabe_mit_tool = Task(
    description="""\
        Recherchiere die AKTUELLSTEN Wahlumfragen fuer Deutschland.

        Suche im Internet nach den neuesten Umfrageergebnissen und beantworte:
        1. Welche Umfragewerte haben CDU/CSU, SPD, Gruene, Linke, FDP, AfD, BSW aktuell?
        2. Welche Partei liegt vorne?
        3. Von wann ist die neueste Umfrage und wer hat sie durchgefuehrt?

        Nenne konkrete Prozentzahlen, das genaue Datum und die Quelle der Umfragen.
    """,
    expected_output="Uebersicht der Wahlumfragen mit Prozentzahlen, Datum und Quellenangabe.",
    agent=agent_mit_tool,
)

ergebnis_mit = Crew(agents=[agent_mit_tool], tasks=[aufgabe_mit_tool]).kickoff()

display(Markdown(f"---\n## Ergebnis: Mit Web-Tool\n\n{ergebnis_mit.raw}"))

---
### Teil C – Agent mit Web-Tool + heutigem Datum ✅✅

**Problem aus Teil B:** Der Agent sucht zwar im Internet, aber er weiss nicht welches Datum heute ist!
Er könnte nach *"aktuellen Umfragen 2024"* suchen, obwohl wir schon viel später sind.

**Lösung:** Wir übergeben das heutige Datum einfach im Task-Text.

> 💡 KI-Agenten wissen nicht was **"heute"** ist – wir müssen es ihnen sagen!

Das ist der gesamte Unterschied zu Teil B:
```python
from datetime import date
HEUTE = date.today().strftime("%d.%m.%Y")
# ... dann im Task:
description=f"""\
    Heute ist der {HEUTE}.
    ...
"""
```

In [ ]:
from datetime import date
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool
from IPython.display import display, Markdown

# Heutiges Datum automatisch ermitteln
HEUTE = date.today().strftime("%d.%m.%Y")
print(f"Heutiges Datum wird übergeben: {HEUTE}")

such_tool = SerperDevTool()
#such_tool = WebsiteSearchTool(website='https://dawum.de/')

agent_mit_datum = Agent(
    role="Politischer Analyst",
    goal="Recherchiere aktuelle Wahlumfragen und politische Stimmungsbilder in Deutschland.",
    backstory="""\
        Du bist ein erfahrener politischer Analyst mit Fokus auf deutsche Politik.
        Du wertest Wahlumfragen aus und erklaerst politische Trends verstaendlich.
        Du nutzt immer aktuelle Quellen und gibst diese an.
    """,
    verbose=False,
    tools=[such_tool],
)

# <-- das ist der einzige Unterschied zu Teil B: die erste Zeile mit dem Datum!
aufgabe_mit_datum = Task(
    description=f"""\
        Heute ist der {HEUTE}.

        Recherchiere die AKTUELLSTEN Wahlumfragen fuer Deutschland aus den letzten 7 Tagen.

        Beantworte:
        1. Welche Umfragewerte haben CDU/CSU, SPD, Gruene, Linke, FDP, AfD, BSW aktuell?
        2. Welche Partei liegt vorne?
        3. Von wann ist die neueste Umfrage und wer hat sie durchgefuehrt?

        Nenne konkrete Prozentzahlen, das genaue Datum und die Quelle.
    """,
    expected_output="Uebersicht der aktuellen Wahlumfragen mit Prozentzahlen, Datum und Quellenangabe.",
    agent=agent_mit_datum,
)

ergebnis_datum = Crew(agents=[agent_mit_datum], tasks=[aufgabe_mit_datum]).kickoff()

display(Markdown(f"---\n## Ergebnis: Mit Web-Tool + Datum ({HEUTE})\n\n{ergebnis_datum.raw}"))

### 💡 Was haben wir gelernt?

| | Teil A: Ohne Tool | Teil B: Mit Tool | Teil C: Mit Tool + Datum |
|---|---|---|---|
| **Datenquelle** | Trainingsdaten (veraltet) | Google-Suche | Google-Suche |
| **Datum bekannt?** | Nein ❌ | Nein ❌ | Ja ✅ |
| **Genauigkeit** | Veraltet | Besser, aber evtl. falsches Jahr | Aktuell |
| **Code-Änderung** | – | `tools=[such_tool]` | + `f"Heute ist der {HEUTE}"` |

**Fazit:** KI-Agenten wissen nicht was **"heute"** ist – wir müssen es ihnen immer explizit sagen!

---
## Sample 3 – Agenten-Zusammenarbeit: Die Debatte

Bisher hat immer nur **ein Agent** eine Aufgabe erledigt.
Jetzt arbeiten **drei Agenten zusammen** – wie ein echtes Team:

| Agent | Rolle |
|---|---|
| **Pro-Anwalt** | Argumentiert FÜR die These |
| **Contra-Anwalt** | Argumentiert GEGEN die These – und antwortet auf Pro |
| **Richter** | Liest beide Seiten und fällt ein Urteil |

### Neues Konzept: `context=[]`
Jeder Agent bekommt die Ergebnisse der vorherigen Agenten als **Kontext** übergeben.
So *liest* der Contra-Agent was der Pro-Agent geschrieben hat, bevor er antwortet.

```python
aufgabe_contra = Task(
    ...،
    context=[aufgabe_pro]  # <-- liest die Pro-Argumente!
)
```

### Ausprobieren:
Ändere `THESE` zu einem anderen Thema – z.B.:
- `"Hausaufgaben sollten abgeschafft werden"`
- `"KI ist gefaehrlicher als nuetzlich"`
- `"Videospiele machen klug"`

In [ ]:
from crewai import Agent, Task, Crew, Process
from IPython.display import display, Markdown

# ── Hier kannst du die These ändern ──────────────────────
THESE = "Jede Familie sollte mindestens ein Haustier haben."
# ─────────────────────────────────────────────────────────

pro_agent = Agent(
    role="Anwalt der Pro-Seite",
    goal=f"Verteidige ueberzeugend die These: '{THESE}'",
    backstory="""\
        Du bist ein brillanter Redner und Debattierer.
        Deine Aufgabe ist es, die beste moegliche Argumentation
        FUER eine These zu liefern - egal was du persoenlich denkst.
        Du verwendest Fakten, Beispiele und Logik.
    """,
    verbose=False,
)

contra_agent = Agent(
    role="Anwalt der Contra-Seite",
    goal=f"Widerlege ueberzeugend die These: '{THESE}'",
    backstory="""\
        Du bist ein brillanter Redner und Debattierer.
        Deine Aufgabe ist es, die beste moegliche Argumentation
        GEGEN eine These zu liefern - egal was du persoenlich denkst.
        Du verwendest Fakten, Beispiele und Logik.
        Du gehst gezielt auf die Argumente der Pro-Seite ein.
    """,
    verbose=False,
)

richter_agent = Agent(
    role="Unparteiischer Richter",
    goal="Bewerte die Debatte fair und faelle ein begruendetes Urteil.",
    backstory="""\
        Du bist ein erfahrener Richter und Mediator.
        Du hoerst beide Seiten einer Debatte und bewertest
        die Qualitaet der Argumente sachlich und fair.
        Du gibst keine persoenliche Meinung ab, sondern
        urteilst allein nach der Staerke der Argumentation.
    """,
    verbose=False,
)

aufgabe_pro = Task(
    description=f"""\
        Die Debattenthese lautet: "{THESE}"

        Liefere 3 starke Argumente FUER diese These.
        Jedes Argument soll:
        - Eine klare Behauptung enthalten
        - Mit einem Beispiel oder Fakt unterstuetzt werden        
    """,
    expected_output="""\
        3 nummerierte Pro-Argumente mit je einer Begruendung
        und einem konkreten Beispiel.
    """,
    agent=pro_agent,
)

aufgabe_contra = Task(
    description=f"""\
        Die Debattenthese lautet: "{THESE}"

        Du hast die Pro-Argumente gelesen. Liefere nun 3 starke
        Argumente GEGEN diese These und gehe dabei gezielt auf
        die Pro-Argumente ein.
        Jedes Argument soll:
        - Eine klare Behauptung enthalten
        - Mit einem Beispiel oder Fakt unterstuetzt werden        
    """,
    expected_output="""\
        3 nummerierte Contra-Argumente mit je einer Begruendung,
        einem konkreten Beispiel und einem direkten Bezug auf
        das jeweilige Pro-Argument.
    """,
    agent=contra_agent,
    context=[aufgabe_pro],  # <-- liest die Pro-Argumente!
)

aufgabe_urteil = Task(
    description=f"""\
        Du hast beide Seiten der Debatte zur These "{THESE}" gelesen.

        Faelle jetzt ein Urteil:
        1. Fasse die staerksten Argumente beider Seiten kurz zusammen
        2. Bewerte welche Seite insgesamt ueberzeugender argumentiert hat
        3. Erklaere warum - bezogen auf Logik und Beweiskraft
        4. Formuliere ein abschliessendes Fazit in 2-3 Saetzen
    """,
    expected_output="""\
        Ein strukturiertes Urteil mit Zusammenfassung beider Seiten,
        einer begruendeten Entscheidung und einem klaren Fazit.
    """,
    agent=richter_agent,
    context=[aufgabe_pro, aufgabe_contra],  # <-- liest beide Seiten!
)

crew = Crew(
    agents=[pro_agent, contra_agent, richter_agent],
    tasks=[aufgabe_pro, aufgabe_contra, aufgabe_urteil],
    process=Process.sequential,
    verbose=False,
)

ergebnis = crew.kickoff()

display(Markdown(f"""
---
## Debatte: {THESE}

### Pro-Argumente
{aufgabe_pro.output.raw}

---
### Contra-Argumente
{aufgabe_contra.output.raw}

---
### Urteil des Richters
{ergebnis.raw}
"""))

### 💡 Was haben wir gelernt?

- **`context=[...]`** gibt einem Agenten die Ergebnisse anderer Agenten als Eingabe
- **`Process.sequential`** stellt sicher dass die Agenten nacheinander arbeiten
- Jeder Agent hat eine klar andere **Perspektive** — obwohl es dieselbe KI ist!
- Das zeigt: KI hat keine echte Meinung, sie nimmt die Rolle an die wir ihr geben

---
## Quiz-Spiel (Vollständiges Beispiel)

Das vollständige Quiz-Spiel aus `quiz_game.py` mit **3 Agenten** die zusammenarbeiten:

| Agent | Aufgabe |
|---|---|
| **Fragen-Ersteller** | Erfindet Multiple-Choice-Fragen zum Thema |
| **Quiz-Master** | Stellt die Fragen und wertet aus |
| **Erklärer** | Erklärt die richtigen Antworten am Ende |

In [ ]:
# Quiz-Spiel direkt aus quiz_game.py ausführen

import os, sys
sys.path.insert(0, os.getcwd())

import importlib
import quiz_game
importlib.reload(quiz_game)
quiz_game.main()

---
## Sample 4 – KI entwickelt ein Spiel

Das komplexeste Beispiel: Mehrere KI-Crews arbeiten zusammen, um ein **Text-Adventure zu entwickeln** – von der Idee bis zum fertigen Python-Code.

### Der Prozess hat 3 Phasen:

| Phase | Wer | Was |
|-------|-----|-----|
| 1 💡 Idee | 1 Agent | Erfindet ein Spielkonzept zum Thema das du wählst |
| 2 🎮 Design | 3 Agenten | Erstellt Räume, Gegenstände, Rätsel, Enden und ASCII-Art |
| 3 👩‍💻 Code | 2 Agenten | Schreibt und reviewed den fertigen Python-Code |

Nach jeder Phase kannst du **Feedback geben** – die KI überarbeitet das Ergebnis so lange, bis du zufrieden bist!

> **Hinweis:** Dieses Sample läuft im Terminal unten (nicht im Notebook).
> Alle Eingaben machst du dort – einfach die Zelle ausführen und das Terminal beobachten!

### Was du lernst:
- Mehrere `Crew`s hintereinander schalten
- **Feedback-Schleife:** Crew läuft erneut bis das Ergebnis passt
- Ergebnisse einer Crew als Eingabe für die nächste nutzen
- KI-generierter Code wird als `.py` Datei gespeichert – mit ASCII-Art pro Raum!


In [ ]:
# ==============================================================================
# Sample 4: KI entwickelt ein Spiel
# ==============================================================================
# Dieses Sample braucht ein echtes Terminal fuer die Eingaben.
# Fuehre den Befehl unten im Terminal aus (Tab "Terminal" in VS Code).
# ==============================================================================

import sys
import os
from IPython.display import display, Markdown

script = os.path.join(os.getcwd(), "sample04_game_dev.py")
python = sys.executable  # derselbe Python-Interpreter der gerade laeuft

display(Markdown(f"""
## Sample 4 starten

Dieses Sample laeuft im Terminal, nicht im Notebook – so funktionieren die Eingaben zuverlaessig.

**So gehts:**
1. Oeffne das Terminal in VS Code (unten, Tab **Terminal**)
2. Tippe diesen Befehl ein und druecke Enter:

```
& "{python}" "{script}"
```

Alle Eingaben und Ergebnisse erscheinen direkt im Terminal.
"""))
